# Interactive LAVA -- step-controlled variant

Like `interactive_lava.ipynb`, but every command carries its own step count.
Call `run_command("move the red block to the left", 10)` to run the low-level
policy for exactly 10 steps with that instruction, or chain commands with
`run_sequence([("move to the red moon", 20), ("push it to the left", 40)])`.

In [ ]:
import sys, os
sys.path.insert(0, os.path.expanduser("~/projects/language-table"))

import nest_asyncio; nest_asyncio.apply()
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML

from language_table.environments.language_table import LanguageTable
from language_table.environments.blocks import LanguageTableBlockVariants
from language_table.environments.rewards.block2block import BlockToBlockReward
from language_table.lamer.lava_policy import LAVAPolicy
from language_table.lamer.state_to_text import state_to_text

In [ ]:
import sys, jax
print("python:", sys.executable)
print("jax:   ", jax.__version__)
print("devices:", jax.devices())

In [ ]:
env = LanguageTable(
    block_mode=LanguageTableBlockVariants("BLOCK_8"),
    reward_factory=BlockToBlockReward,
    seed=42,
)

policy = LAVAPolicy(
    checkpoint_dir="/home/mateo/projects/Isaac/language-table/checkpoints",
    checkpoint_prefix="bc_resnet_sim_checkpoint_955000",
    action_clip=0.1,
)
print("Env and policy ready.")

In [ ]:
DEFAULT_STEPS = 96


def run_command(command: str, steps: int = DEFAULT_STEPS, *, reset_policy: bool = True, verbose: bool = True):
    """Run the low-level policy with *command* for exactly *steps* steps.

    Returns (frames, total_reward, info). `reset_policy=False` preserves the
    policy's internal state so `run_sequence` can chain commands without
    clearing action history between segments.
    """
    if reset_policy:
        policy.reset(num_envs=1)

    obs = env._compute_observation()
    frames = [env.render(mode="rgb_array")]
    total_reward = 0.0
    info = {}

    if verbose:
        print(f"Current state:\n{state_to_text(env._last_state)}\n")
        print(f"Command: {command}  ({steps} steps)\n")

    for _ in range(steps):
        actions = policy.predict(
            goals=[command],
            obs_list=[obs],
            active_mask=np.array([True]),
        )
        action = actions[0]

        # Ignore env `done` -- fires on the env's randomly-sampled preset
        # task, not on our command. Always run the full `steps`.
        obs, reward, _done, info = env.step(action)
        frames.append(env.render(mode="rgb_array"))
        total_reward += reward

    if verbose:
        print(f"\nTotal reward: {total_reward:.3f}  |  Won: {info.get('won', False)}")
        print(f"Final state:\n{state_to_text(env._last_state)}")
    return frames, total_reward, info


def _parse_spec(spec, default_steps=DEFAULT_STEPS):
    """Accept a string, (command, steps) tuple, or (command,) tuple."""
    if isinstance(spec, str):
        return spec, default_steps
    if len(spec) == 1:
        return spec[0], default_steps
    command, steps = spec
    return command, int(steps)


def run_sequence(specs, *, verbose: bool = True):
    """Run a list of (command, steps) pairs back-to-back without resetting the env.

    Each spec may be either a `(command, steps)` tuple or a bare command
    string (which uses DEFAULT_STEPS). The policy is reset once at the start
    of the sequence; across commands it keeps its internal state so chained
    instructions flow smoothly.
    """
    policy.reset(num_envs=1)
    all_frames = []
    total_reward = 0.0
    info = {}
    boundaries = [0]

    for i, spec in enumerate(specs):
        command, steps = _parse_spec(spec)
        if verbose:
            print(f"[seq {i}] {command!r}  ({steps} steps)")
        frames, reward, info = run_command(
            command, steps=steps, reset_policy=False, verbose=False,
        )
        # Drop the first frame of every follow-up segment -- duplicates the
        # last frame of the previous one.
        if all_frames:
            frames = frames[1:]
        all_frames.extend(frames)
        total_reward += reward
        boundaries.append(len(all_frames))

    if verbose:
        print(f"\nTotal reward: {total_reward:.3f}  |  Won: {info.get('won', False)}")
        print(f"Final state:\n{state_to_text(env._last_state)}")
    return all_frames, total_reward, info, boundaries


def show_rollout(frames, fps=5, title=None):
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.axis("off")
    img = ax.imshow(frames[0])
    ax.set_title(title or f"Rollout  ({len(frames)} frames)")

    def update(i):
        img.set_data(frames[i])
        return [img]

    ani = animation.FuncAnimation(
        fig, update, frames=len(frames),
        interval=1000 / fps, blit=True,
    )
    plt.close(fig)
    return HTML(ani.to_jshtml())

In [ ]:
env.reset()
state = env._last_state

print(state_to_text(state))

plt.figure(figsize=(5, 5))
plt.imshow(env.render(mode="rgb_array"))
plt.axis("off")
plt.title("Initial state")
plt.tight_layout()
plt.show()

## Single command with step budget

Pass the number of steps as the second positional arg, or as `steps=`.

In [ ]:
env.reset()
frames, reward, info = run_command("move the red block to the left", 10)
show_rollout(frames)

## Chained commands, each with its own step budget

In [ ]:
env.reset()
plan = [
    ("move to the red moon", 20),
    ("push the red moon to the left", 40),
    ("move to the blue cube", 20),
]
frames, reward, info, boundaries = run_sequence(plan)
print("segment boundaries (frame indices):", boundaries)
show_rollout(frames)